In [1]:
# Check if a GPU is available
!nvidia-smi || echo 'No GPU available. You may be on a CPU-only runtime.'

Tue Dec 23 13:27:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models


In [5]:
import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

Ollama server started as background process (PID: 843 )
Waited 5 seconds for the server to initialize.


In [6]:
# LLM
!ollama pull gpt-oss:20b

# SLM
!ollama pull gemma3:1b

# Embedding
!ollama pull nomic-embed-text

In [7]:
!pip install -r '/content/drive/MyDrive/RAG/requirements.txt'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 149.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
from pathlib import Path
import sys
import time
from pprint import pprint

# Add src to path
sys.path.append(str(Path('/content/drive/MyDrive/RAG/src').parent))

from src.config.config import Config
from src.document_ingestion.document_processor import DocumentProcessor
from src.vectorstore.vectorstore import VectorStore
from src.graph_builder.graph_builder_adv import GraphBuilder

In [10]:
import os
import shutil

llm = Config.get_llm()
embedding = Config.get_embedding_model()

# Define the expected Chroma persistence directory
chroma_persist_dir = "/content/drive/MyDrive/RAG/chroma_db"

# Remove any existing Chroma persistence directory to avoid dimension conflicts
if os.path.exists(chroma_persist_dir):
    shutil.rmtree(chroma_persist_dir)
    print(f"Removed existing Chroma persistence directory: {chroma_persist_dir}")

doc_processor = DocumentProcessor(embedding)
vector_store = VectorStore(embedding)

pdfs = '/content/drive/MyDrive/RAG/datasets'

document_processor = DocumentProcessor(embedding)
documents = document_processor.process(pdfs)

# Create vector store
vector_store.create_vectorstore(documents)


Removed existing Chroma persistence directory: /content/drive/MyDrive/RAG/chroma_db


InvalidArgumentError: Collection expecting embedding with dimension of 384, got 768

In [ ]:
# Build graph
graph_builder = GraphBuilder(
    retriever=vector_store.get_retriever(),
    llm=llm
)
builder = graph_builder.build() # Assign the returned workflow to a variable

In [ ]:
%%time
import os
import math
import pandas as pd

# Ensure dataset is loaded if not already defined (in case previous cell was skipped)
if 'dataset' not in locals():
    # Assuming the path is correct as per accdd6c1
    dataset = pd.read_excel('/content/drive/MyDrive/RAG/datasets/datasets.xlsx', sheet_name="final")

answers = {}

for i, q in enumerate(dataset['question'].tolist()):
    # Skip NaN values in the question list
    if isinstance(q, float) and math.isnan(q):
        print(f"Skipping question at index {i} due to NaN value.")
        continue

    response = graph_builder.run(q)
    answers[i] = {} # Initialize the dictionary for the current query index
    answers[i]['query'] = response['question']
    answers[i]['answer'] = response['answer']
    answers[i]['retrieved_docs'] = [] # Initialize retrieved_docs as a list of dictionaries

    for doc in response['retrieved_docs']:
        answers[i]['retrieved_docs'].append({
            'content': doc.page_content,
            'doc_name': doc.metadata.get('title', os.path.basename(doc.metadata['source'])),
            'page': doc.metadata['page'],
            'page_label': doc.metadata['page_label'],
            'source': doc.metadata['source']
        })


In [ ]:
import pandas as pd

data_for_df = []
for idx, item in answers.items():
    base_row = {
        'index': idx,
        'query': item['query'],
        'answer': item['answer']
    }

    # Check if 'retrieved_docs' is a list and not empty
    if 'retrieved_docs' in item and isinstance(item['retrieved_docs'], list) and item['retrieved_docs']:
        for doc_info in item['retrieved_docs']:
            row = base_row.copy()
            row.update({
                'content': doc_info.get('content'),
                'doc_name': doc_info.get('doc_name'),
                'page': doc_info.get('page'),
                'page_label': doc_info.get('page_label'),
                'source': doc_info.get('source')
            })
            data_for_df.append(row)
    else:
        # Handle cases where retrieved_docs might be missing, not a list, or empty
        row = base_row.copy()
        row.update({
            'content': None,
            'doc_name': None,
            'page': None,
            'page_label': None,
            'source': None
        })
        data_for_df.append(row)

df_answers = pd.DataFrame(data_for_df)
display(df_answers.head())

In [ ]:
merged_df_by_qa = pd.merge(dataset, df_answers, left_on='question', right_on='query', how='left')
display(merged_df_by_qa.head())

In [ ]:
merged_df_by_qa.to_csv('/content/drive/MyDrive/RAG/datasets/adv_rag_with_gpt_oss_result_20251223.csv', index=False)